In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

customers = spark.createDataFrame([
    (1,"John ","Hyderabad"),
    (2,"Alice","Chennai"),
    (3,None,"Bangalore")
],["customer_id","name","city"])

cars = spark.createDataFrame([
    (101,"Toyota","Camry",30000),
    (102,"Honda","Civic",-20000),
    (103,"Hyundai","i20",15000)
],["car_id","brand","model","price"])

sales = spark.createDataFrame([
    (1,1,101,"2024-01-01",1),
    (2,2,102,"2024-01-02",2),
    (3,99,103,"2024-01-03",1)
],["sale_id","customer_id","car_id","sale_date","quantity"])

sales = sales.withColumn("sale_date", to_date(col("sale_date")))

df = sales.join(customers,"customer_id").join(cars,"car_id")
df.groupBy("customer_id").sum("price").show()
cars.display()
customers.display()
sales.display()

In [0]:
#handle nulls 
customers_cleaned = customers.fillna({"name": "Unknown"})
customers_cleaned.display()

In [0]:
#handle negative prices
cars_cleaned = cars.withColumn("price", abs(col("price")))
cars_cleaned.display()

In [0]:
# removing invalid keys 
df_final = sales.join(customers_cleaned, "customer_id", "inner") \
                .join(cars_cleaned, "car_id", "inner")

df_final.display()

In [0]:
customer_count_before_cleaning=customers.count()
customer_count_after_cleaning=customers_cleaned.count()
print("Number of customers before cleaning: ",customer_count_before_cleaning)
print("Number of customers after cleaning: ",customer_count_after_cleaning)

In [0]:
from pyspark.sql.functions import col

# 1. Anti-Joins (The "Garbage" collector)
# This finds sales that don't have a matching customer
bad_customer_data = sales.join(customers, ["customer_id"], "left_anti")

# This finds sales that don't have a matching car
bad_car_data = sales.join(cars, ["car_id"], "left_anti")

# 2. Count the validation metrics
total_raw = sales.count()
invalid_fk = bad_customer_data.count() + bad_car_data.count()
final_records = df_final.count() # assuming df_final is your inner join result

# 3. Print the Validation Report
print("-" * 30)
print("PHASE 3: VALIDATION REPORT")
print("-" * 30)
print(f"Total Raw Records:      {total_raw}")
print(f"Invalid Foreign Keys:   {invalid_fk}")
print(f"Final Cleaned Records:  {final_records}")
print("-" * 30)

# 4. Explain the drop (Comparison)
if (total_raw - invalid_fk) == final_records:
    print("SUCCESS: Data counts match. Validation passed.")
else:
    print("WARNING: Count mismatch! Check your join logic.")

In [0]:
customers_cleaned.display()
cars_cleaned.display()
df_final.display()

In [0]:
# revenue per each customer 
from pyspark.sql.functions import sum, count, desc


revenue_per_cust = df_final.groupBy("customer_id", "name") \
    .agg(sum("price").alias("total_spent")) \
    .orderBy(desc("total_spent"))

revenue_per_cust.display()

In [0]:
## cars per brand 
cars_per_brand = df_final.groupBy("brand") \
    .agg(count("car_id").alias("cars_sold")) \
    .orderBy(desc("cars_sold"))

revenue_per_cust.display()

In [0]:
# city wise revenue 
city_revenue = df_final.groupBy("city") \
    .agg(sum("price").alias("city_total_revenue")) \
    .orderBy(desc("city_total_revenue"))

city_revenue.display()

In [0]:
df_final.createOrReplaceTempView("final_sales") 

In [0]:

%sql
SELECT city, brand, model, sales_count
FROM (
  SELECT 
    city, brand, model, 
    COUNT(*) as sales_count,
    DENSE_RANK() OVER (PARTITION BY city ORDER BY COUNT(*) DESC) as rank
  FROM final_sales
  GROUP BY city, brand, model
)
WHERE rank <= 3

In [0]:
%sql
SELECT customer_id, name, COUNT(*) as purchase_count
FROM final_sales
GROUP BY customer_id, name
ORDER BY purchase_count DESC

In [0]:
%sql
SELECT 
  TRUNC(CAST(sale_date AS DATE), 'MM') as month,
  SUM(price) as monthly_revenue,
  COUNT(sale_id) as total_transactions
FROM final_sales
GROUP BY month
ORDER BY month ASC

In [0]:
%sql
DESCRIBE final_sales